<a href="https://colab.research.google.com/github/diyanrahma/Retrieval-Augmented-Generation-Automatic-Fact-Checking-PubHealth-Dataset/blob/main/Metode_Question_Generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch==2.1.2 torchvision==0.16.2 torchaudio==2.1.2
!pip install transformers==4.37.2 datasets sentence-transformers accelerate

In [ ]:
from transformers import AutoTokenizer
import pandas as pd
import re
import torch
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [ ]:
df_raw = pd.read_excel("/content/pubhealth_biomed_only.xlsx")
df_raw.head()

In [ ]:
ds = df_raw.to_dict(orient="records")

In [ ]:
def norm(x):
    return {
        "id": x.get("id"),
        "claim": (x.get("claim") or "").strip(),
        "label": str(x.get("label")).strip().lower()
    }

df = pd.DataFrame([norm(i) for i in ds])
df = df[df["claim"].str.len() > 0].reset_index(drop=True)
df.head()

In [ ]:
tok = AutoTokenizer.from_pretrained("google/flan-t5-base")
mdl = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

In [ ]:
def clean_claim(text):
    if not isinstance(text, str):
        text = str(text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'\"+', '', text)
    return text

In [ ]:
def qg_strict(claim):
    prompt = (
        "Rewrite the statement into a single clear YES/NO question.\n"
        "Avoid quotes and boilerplate. Keep under 15 words.\n"
        f"Statement: {clean_claim(claim)}\nQuestion:"
    )
    ids = tok(prompt, return_tensors="pt", truncation=True).input_ids
    with torch.no_grad():
        out = mdl.generate(
            ids,
            max_new_tokens=24,
            num_beams=6,
            do_sample=False,
            length_penalty=0.1,
            repetition_penalty=1.2,
        )
    q = tok.decode(out[0], skip_special_tokens=True).strip()
    q = re.sub(r'\"+', '', q).strip()
    if not q.endswith("?"):
        q += "?"
    return q

df_qg = df.head(100).copy()
df_qg["question"] = df_qg["claim"].apply(qg_strict)
df_qg[["id", "question", "label"]].head()

In [ ]:
df_qg = df.head(100).copy()
df_qg["question"] = df_qg["claim"].apply(qg_strict)

df_qg = df_qg[["id", "question", "label"]]

df_qg.head()

In [ ]:
yesno_starts = ("is","are","do","does","did","can","will","was","were","should")
wh_starts = ("what","who","when","where","why","how")

def classify_question(q):
    if not isinstance(q, str):
        return "unknown"

    q_lower = q.strip().lower()
    first_word = q_lower.split()[0]

    if first_word in wh_starts:
        return "wh"
    elif first_word in yesno_starts:
        return "yes/no"
    else:
        return "yes/no"

df_qg["question_type"] = df_qg["question"].apply(classify_question)

distribution = df_qg["question_type"].value_counts()

print(distribution)

In [ ]:
!pip install wordcloud matplotlib

from wordcloud import WordCloud, STOPWORDS
import matplotlib.pyplot as plt
import re

In [ ]:
text = " ".join(df_qg["question"].astype(str).tolist())

text = re.sub(r'[^a-zA-Z\s]', '', text)
text = text.lower()

In [ ]:
custom_stopwords = set(STOPWORDS)
custom_stopwords.update([
    "is", "are", "was", "were",
    "do", "does", "did",
    "can", "could", "should",
    "has", "have", "had",
    "will", "would", "may", "might", "must"
])

wc = WordCloud(
    width=800,
    height=400,
    background_color="white",
    stopwords=custom_stopwords,
    colormap="viridis"
).generate(text)

plt.figure(figsize=(12,6))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.title("Word Cloud of Generated Questions")
plt.show()

In [ ]:
def normalize_question(text):
    if not isinstance(text, str):
        text = str(text)

    text = text.lower()
    text = re.sub(r'\"+', '', text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^a-z\s?]', '', text)
    text = text.strip()

    return text

df_qg["question_normalized"] = df_qg["question"].apply(normalize_question)

df_qg.head()